# Sentiment Analysis system using ML + DL + NLP

In [29]:
import pandas as pd
import numpy as np
import nltk
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [30]:
df = pd.read_csv("All DataSets/IMDB Dataset.csv")  # columns: review, sentiment
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [31]:
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# NLP Preprocessing

In [32]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

# Check if words exist in stop_words before removing
if 'not' in stop_words:
    stop_words.remove('not')
if 'no' in stop_words:
    stop_words.remove('no')
if 'never' in stop_words:
    stop_words.remove('never')
# Alternative approach: stop_words.discard('never') would not raise an error if 'never' is not in the set

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\91839\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [33]:
df['cleaned_review'] = df['review'].apply(clean_text)

# Train-Test Split

In [34]:
X = df['cleaned_review']
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Machine Learning Model (TF-IDF + Logistic Regression)

In [13]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model_ml = LogisticRegression()
model_ml.fit(X_train_tfidf, y_train)

y_pred_ml = model_ml.predict(X_test_tfidf)

print("ML Accuracy:", accuracy_score(y_test, y_pred_ml))

ML Accuracy: 0.885


# Deep Learning Model (LSTM)

In [35]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=100)
X_test_pad = pad_sequences(X_test_seq, maxlen=100)

# Model:

In [36]:
model_dl = Sequential()
model_dl.add(Embedding(input_dim=5000, output_dim=64))
model_dl.add(LSTM(64))
model_dl.add(Dense(1, activation='sigmoid'))

model_dl.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model_dl.fit(X_train_pad, y_train, epochs=5, batch_size=32)

Epoch 1/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 40s 29ms/step - accuracy: 0.8478 - loss: 0.3482
Epoch 2/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 36s 29ms/step - accuracy: 0.8985 - loss: 0.2495
Epoch 3/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 36s 28ms/step - accuracy: 0.9156 - loss: 0.2102
Epoch 4/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 41s 29ms/step - accuracy: 0.9332 - loss: 0.1696
Epoch 5/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 36s 29ms/step - accuracy: 0.9469 - loss: 0.1378


# Evaluation

In [37]:
y_pred_dl = (model_dl.predict(X_test_pad) > 0.5).astype("int32")

print("DL Accuracy:", accuracy_score(y_test, y_pred_dl))
print(classification_report(y_test, y_pred_dl))

313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step
DL Accuracy: 0.8644
              precision    recall  f1-score   support

           0       0.86      0.86      0.86      4932
           1       0.87      0.87      0.87      5068

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000



# Prediction Function (for Deployment )

In [38]:
def predict_sentiment(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=100)
    pred = model_dl.predict(pad)[0][0]
    
    if pred > 0.5:
        return "Positive 😊"
    else:
        return "Negative 😠"

# Streamlit App

#Streamlit application code is written and executed in VS Code.

In [39]:
import streamlit as st
import pickle
from tensorflow.keras.preprocessing.sequence import pad_sequences
import re

# load model & tokenizer
model = pickle.load(open("model.pkl", "rb"))
tokenizer = pickle.load(open("tokenizer.pkl", "rb"))

st.title("Sentiment Analysis App")

user_input = st.text_area("Enter your text:")

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

def predict_sentiment(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=100)
    pred = model.predict(pad)[0][0]
    return "Positive 😊" if pred > 0.5 else "Negative 😠"

if st.button("Predict"):
    st.write("Sentiment:", predict_sentiment(user_input))

2026-03-24 19:43:12.115 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 19:43:12.116 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 19:43:12.117 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 19:43:12.119 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 19:43:12.122 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 19:43:12.124 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 19:43:12.125 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-24 19:43:12.127 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [40]:
import pickle

# model save
pickle.dump(model_dl, open("model.pkl", "wb"))

# tokenizer save
pickle.dump(tokenizer, open("tokenizer.pkl", "wb"))